In [ ]:
from typing import TypedDict


class State(TypedDict):
    topic: str
    story: str


import os

import dotenv
from langchain.chat_models import init_chat_model

dotenv.load_dotenv()
model = init_chat_model(
    model="deepseek-v4-pro",
    base_url="https://api.deepseek.com/v1",
    api_key=os.getenv("DEEPSEEK_API"),
    model_provider="openai",
    temperature=0.7,
)

from langchain.messages import AIMessage, HumanMessage


def call_model(state: State) -> State:
    model_response: AIMessage = model.invoke([HumanMessage(content=f"生成一个关于 {state['topic']} 的故事")])
    print(type(model_response.content))
    return {"story": model_response.content}


from langgraph.graph import END, START, StateGraph

graph = (
    StateGraph(State)
    .add_node("call_model", call_model)
    .add_edge(START, "call_model")
    .add_edge("call_model", END)
    .compile()
)

async for msg, metadata in graph.astream({"topic": "公主"}, stream_mode="messages"):
    # print(msg.content, end="")
    print(metadata)
    pass

<class 'str'>
